# 모의고사 05. 고객 이탈 예측 (이진분류, 불균형 데이터)

> 실제 시험처럼 이론 설명 없이 문제만 순서대로 제시합니다. **90분 타이머를 맞추고** 풀어보세요. 오픈북 허용 문서(numpy/pandas/matplotlib/seaborn/scikit-learn/tensorflow/XGBoost 공식문서)만 참고할 수 있다고 가정합니다.

### 데이터 소개
`data/telco_churn.csv` - 통신사 고객 정보로 이탈(Churn) 여부를 예측합니다. `TotalCharges`에 숨은 결측치(공백 문자열)가 있고, `Churn` 클래스가 불균형(26.5%:73.5%)합니다.

> ⏱️ **[실전 타임어택 가이드]** 데이터 탐색 10분 ➔ 전처리 20분 ➔ 머신러닝 30분 ➔ 딥러닝 30분
> 막히는 부분은 주석으로 남기고 다음 문제로 넘어가는 것이 합격 전략입니다.


## 목차
1교시 데이터 로딩 & EDA · 2교시 데이터 시각화 · 3교시 데이터 전처리 · 4교시 머신러닝 모델링 · 5교시 딥러닝 모델링 · 채점 가이드

In [ ]:
import sys
sys.path.insert(0, '../00_시작하기')
import aice_grader as grader

# 실전처럼 시간 제한(90분)을 두고 풀어보세요. 아래 셀을 실행하면 타이머가 시작됩니다.
timer = grader.ExamTimer(minutes=90)
timer.start()

## 1교시: 데이터 로딩 & EDA

**문제 1.** `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`을 각각 `pd`, `np`, `plt`, `sns`로 불러오고, `../data/telco_churn.csv`을 `df`로 불러와 shape을 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/telco_churn.csv')
print(df.shape)
```

</details>

**문제 2.** `df.info()`로 `TotalCharges`의 타입을 확인하고, 왜 object인지 원인을 코드로 찾아보세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
df.info()  # TotalCharges가 숫자여야 하는데 object(문자열) 타입으로 나옴
print((df['TotalCharges'] == ' ').sum(), '개의 공백 문자열')  # 결측치가 NaN이 아니라 공백 문자열(' ')로 숨어있어 isnull()로는 안 잡힘
```

</details>

**문제 3.** `Churn` 값 분포를 비율(`normalize=True`)로 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print(df['Churn'].value_counts(normalize=True))  # normalize=True로 개수 대신 비율을 확인해 불균형 정도를 파악  # 범주형 데이터의 항목별 개수를 세어 데이터 분포 확인
```

</details>

## 2교시: 데이터 시각화

**문제 4.** `Churn` 분포를 countplot으로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.countplot(x='Churn', data=df)  # 범주형 변수의 항목별 개수를 막대 그래프로 시각화
plt.title('Churn (imbalanced)')
plt.show()
```

</details>

**문제 5.** `Contract`별 `tenure` 분포를 boxplot으로 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.boxplot(x='Contract', y='tenure', data=df)  # 이상치와 사분위수를 확인하기 위해 상자 수염 그림 시각화
plt.show()
```

</details>

## 3교시: 데이터 전처리

**문제 6.** `TotalCharges`의 공백을 NaN으로 바꾼 뒤 float로 변환하고 중앙값으로 결측치를 채우세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan).astype(float)  # 공백을 NaN으로 바꿔야 비로소 float 변환이 가능해짐
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())  # 이상치 영향이 적은 중앙값으로 결측치 대체  # 결측치를 지정한 값(평균, 중앙값 등)으로 안전하게 대체
```

</details>

**문제 7.** `customerID`를 삭제하고, `Churn`을 0/1로 변환하고, 나머지 object 컬럼을 `get_dummies(drop_first=True)`로 인코딩하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
df = df.drop(columns=['customerID'])
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
cat_cols = df.select_dtypes(include='object').columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)  # 문자로 된 범주형 데이터를 기계가 이해할 수 있는 0과 1 형태(원-핫 인코딩)로 변환
df.info()
```

</details>

**문제 8.** `train_test_split(test_size=0.2, stratify=y, random_state=42)`로 분할하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.model_selection import train_test_split
X = df.drop(columns=['Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)  # 데이터를 모델 학습용(train)과 검증용(test)으로 분리
print(X_train.shape)
```

</details>

## 4교시: 머신러닝 모델링

**문제 9.** `RandomForestClassifier(class_weight='balanced', random_state=42)`를 학습시키고 recall, precision을 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, precision_score
scaler = StandardScaler()  # 데이터를 평균 0, 표준편차 1이 되도록 스케일링 (거리 기반, 딥러닝 알고리즘에 필수)
X_train_s = scaler.fit_transform(X_train)  # 훈련 데이터로 스케일링 기준을 학습(fit)하고 변환(transform)까지 수행
X_test_s = scaler.transform(X_test)  # 훈련 데이터에서 학습된 기준을 그대로 적용하여 평가 데이터를 변환 (데이터 누수 방지)
rf = RandomForestClassifier(class_weight='balanced', random_state=42)  # 여러 개의 의사결정 나무를 묶어 예측하는 랜덤 포레스트 분류 모델 생성
rf.fit(X_train_s, y_train)
pred = rf.predict(X_test_s)
print(recall_score(y_test, pred))
print(precision_score(y_test, pred))
```

</details>

**문제 10.** `imblearn.SMOTE`로 오버샘플링한 뒤 같은 모델을 다시 학습시켜 recall 변화를 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_over, y_over = smote.fit_resample(X_train_s, y_train)
rf2 = RandomForestClassifier(random_state=42)  # 여러 개의 의사결정 나무를 묶어 예측하는 랜덤 포레스트 분류 모델 생성
rf2.fit(X_over, y_over)
print(recall_score(y_test, rf2.predict(X_test_s)))
```

</details>

## 5교시: 딥러닝 모델링

**문제 11.** 출력층 1노드(sigmoid)인 Sequential 모델에 Dropout(0.3)을 포함해서 만들고 `binary_crossentropy`로 compile하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
model = Sequential()  # 딥러닝 층(Layer)을 순서대로 쌓기 위한 기본 틀(모델) 생성
model.add(Dense(32, activation='relu', input_shape=(X_train_s.shape[1],)))  # 이전 층의 모든 노드와 연결되는 완전 연결층(Dense Layer) 추가
model.add(Dropout(0.3))  # 학습 중 일부 노드를 무작위로 꺼서 과적합을 방지
model.add(Dense(16, activation='relu'))  # 이전 층의 모든 노드와 연결되는 완전 연결층(Dense Layer) 추가
model.add(Dense(1, activation='sigmoid'))  # 이진분류이므로 출력 노드 1개 + sigmoid
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])  # 모델 학습 과정(최적화 기법, 손실 함수, 평가지표) 설정
```

</details>

**문제 12.** SMOTE로 오버샘플링한 데이터로 `EarlyStopping(patience=5)`을 사용해 학습시키고 recall을 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from tensorflow.keras.callbacks import EarlyStopping
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)  # 성능이 개선되지 않으면 무의미한 학습을 일찍 멈춤
model.fit(X_over, y_over, epochs=50, batch_size=32, validation_data=(X_test_s, y_test), callbacks=[es], verbose=0)
pred_label = (model.predict(X_test_s).flatten() > 0.5).astype(int)
print(recall_score(y_test, pred_label))
```

</details>

In [ ]:
# 문제를 다 풀었다면 아래 셀을 실행해 소요 시간을 확인하세요.
timer.finish()

## 채점 가이드 (100점 만점 배점표)

이 모의고사는 총 12문제, 100점 만점입니다. 각 문제를 정답과 비교해 맞았으면 배점만큼, 틀렸으면 0점으로 스스로 채점해보세요.

| 문항 | 배점 |
|---|---|
| 문제 1 | 8점 |
| 문제 2 | 8점 |
| 문제 3 | 8점 |
| 문제 4 | 8점 |
| 문제 5 | 8점 |
| 문제 6 | 8점 |
| 문제 7 | 8점 |
| 문제 8 | 8점 |
| 문제 9 | 9점 |
| 문제 10 | 9점 |
| 문제 11 | 9점 |
| 문제 12 | 9점 |
| **합계** | **100점** |

> 💡 **합격 기준: 80점 이상** (실제 AICE Associate와 동일한 기준입니다)

### 자동으로 기록하며 채점하고 싶다면

`00_시작하기/aice_grader.py`의 `MockExamGrader`를 사용하면 점수를 자동으로 합산해줍니다.

```python
import aice_grader as grader

g = grader.MockExamGrader("모의고사05_고객이탈예측")
g.check(1, points=8, is_correct=True)   # 문제 1을 맞았으면 True, 틀렸으면 False
g.check(2, points=8, is_correct=True)
# ... 문제 수만큼 반복 ...
g.report()   # 최종 점수와 합격 여부를 출력
```

### 개념 체크리스트

다 풀었다면 아래 체크리스트로 한 번 더 점검해보세요.

- [ ] 라이브러리를 정확한 alias(pd, np, plt, sns 등)로 불러왔는가
- [ ] 숨은 결측치(공백 문자열)를 찾아냈는가
- [ ] 불균형 데이터에서 accuracy 대신 recall/precision을 확인했는가
- [ ] class_weight 또는 SMOTE 중 하나 이상을 적용했는가
